# 02 · Project 2: SmolLM3-3B-**Base** → chat assistant (SFT) → aligned assistant (DPO), with QLoRA
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/57Ajay/modeltune/blob/main/notebooks/02_project2_sft_dpo_qlora.ipynb)

Companion to [chapter 10](https://github.com/57Ajay/modeltune/blob/main/course/10_preference_optimization.md) and [chapter 11](https://github.com/57Ajay/modeltune/blob/main/course/11_project2_base_to_aligned.md).
This is the modern post-training pipeline in miniature: **base → SFT → preference optimization**.

* Stage 1 config: [`configs/p2_smollm3_sft_qlora.yaml`](https://github.com/57Ajay/modeltune/blob/main/configs/p2_smollm3_sft_qlora.yaml), about 45–70 min on a T4 (estimate)
* Stage 2 config: [`configs/p2_smollm3_dpo_qlora.yaml`](https://github.com/57Ajay/modeltune/blob/main/configs/p2_smollm3_dpo_qlora.yaml), about 50–80 min on a T4 (estimate)

Both stages push their adapter to the Hub, so if Colab disconnects you resume from there.
If your session is short, lower `data.max_train_samples`.

## 0. Setup
Runtime → Change runtime type → **T4 GPU**. Any newer GPU works too; the code picks fp16 or bf16 by itself.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# Get the course code, then install the pinned libraries from pyproject.toml with uv.
# Colab already ships a CUDA build of PyTorch; `uv pip install --system` keeps it instead of downloading another.
!test -d modeltune || git clone -q https://github.com/57Ajay/modeltune.git
%cd modeltune
!git pull -q
!command -v uv >/dev/null || pip install -q uv
!uv pip install --system -q -r pyproject.toml

In [ ]:
# Hugging Face token: add HF_TOKEN in Colab's key icon (Secrets) panel, with WRITE access to push models.
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from huggingface_hub import login
    login()
# Pushing to someone else's namespace fails: change hub.repo_id in the configs (or override it) to YOUR username.

## 1. Meet the base model
It has no chat template, so we give it one (`templates/chatml.jinja`). It has never been taught to *answer*, only to *continue text*. Watch it ramble, or continue the conversation as both sides.

In [ ]:
!python -m modeltune.chat configs/p2_smollm3_sft_qlora.yaml --base --max_new_tokens 200 --prompt "Give me three tips for writing clean Python code." 

## 2. Baseline score
We score answers to the same 100 held-out prompts with a small reward model (Skywork-Reward-V2-Qwen3-0.6B). Higher = more preferred. Also watch `avg_answer_words`.

In [ ]:
!python -m modeltune.evaluate configs/p2_smollm3_sft_qlora.yaml --base

## 3. Stage 1: supervised fine-tuning on chat data
In the printed example, only the assistant turns are in the loss block (`assistant_only_loss` + the `{% generation %}` tags in our template).

In [ ]:
!python -m modeltune.train configs/p2_smollm3_sft_qlora.yaml

In [ ]:
!python -m modeltune.evaluate configs/p2_smollm3_sft_qlora.yaml

In [ ]:
!python -m modeltune.chat configs/p2_smollm3_sft_qlora.yaml --max_new_tokens 300 --prompt "Give me three tips for writing clean Python code." 

## 4. Stage 2: DPO on preference pairs
Starts from the SFT adapter; a frozen copy of it is the reference model. In the logs watch `rewards/accuracies` (should rise above 0.5) and `rewards/margins` (should grow).

In [ ]:
!python -m modeltune.train configs/p2_smollm3_dpo_qlora.yaml

In [ ]:
!python -m modeltune.evaluate configs/p2_smollm3_dpo_qlora.yaml

In [ ]:
!python -m modeltune.chat configs/p2_smollm3_dpo_qlora.yaml --max_new_tokens 300 --prompt "Give me three tips for writing clean Python code." 

## 5. Compare all three

In [ ]:
import json, pandas as pd
paths = {"base": "outputs/p2-smollm3-3b-sft-qlora/eval_base.json",
         "sft": "outputs/p2-smollm3-3b-sft-qlora/eval_finetuned.json",
         "sft+dpo": "outputs/p2-smollm3-3b-dpo-qlora/eval_finetuned.json"}
pd.DataFrame({k: json.load(open(v))["results"] for k, v in paths.items()}).round(3)

## Experiments
1. `train.beta=0.01` vs `0.5` in the DPO stage: more or less drift from the SFT model? Look at answer length.
2. DPO **without** SFT first: set `model.adapter=null` in the DPO config (fresh LoRA on the base model). What happens?
3. `train.loss_type=ipo`, or try `apo_zero` (the SmolLM3 team used APO). Same data, different objective.
4. If the DPO model is much longer but only slightly higher-reward, you've met **length exploitation** (chapter 10).